In [5]:
import os
import csv
import re
from datetime import datetime


def find_lost_communication(root_folder: str, output_csv: str):
    root_folder = os.path.abspath(root_folder)

    timestamp_re = re.compile(
        r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)"
    )

    lost_re = re.compile(
        r"lost communication with\s+(x[\w\d]+)",
        re.IGNORECASE
    )

    exit_status_re = re.compile(
        r"exit_status\s*=\s*(-?\d+)",
        re.IGNORECASE
    )

    time_format = "%Y-%m-%d %H:%M:%S.%f"

    try:
        with open(output_csv, "w", newline="") as out:
            writer = csv.writer(out)
            writer.writerow([
                "date",
                "job",
                "timestamp",
                "Last_lost_communication_timestamp",
                "Job_endtime",
                "Elapsed_time_after_last_lostcomm(min)",
                "Lost communication with",
                "Kill_job?",
                "Exit_status",
                "message"
            ])

            for day_name in sorted(os.listdir(root_folder)):
                day_path = os.path.join(root_folder, day_name)
                if not os.path.isdir(day_path):
                    continue

                for job_name in os.listdir(day_path):
                    job_path = os.path.join(day_path, job_name)
                    if not os.path.isdir(job_path):
                        continue

                    file_path = os.path.join(job_path, "jobtrace")
                    if not os.path.isfile(file_path):
                        continue

                    try:
                        with open(file_path, "r", errors="ignore") as f:
                            lines = f.readlines()

                        lost_lines = [
                            line for line in lines
                            if "lost communication with" in line.lower()
                        ]

                        if not lost_lines:
                            continue

                        kill_job_found = any(
                            "kill_job" in line.lower()
                            for line in lines
                        )
                        kill_status = "Yes" if kill_job_found else "No"

                        exit_statuses = []
                        job_endtime = None

                        for line in lines:
                            # Collect exit statuses
                            matches = exit_status_re.findall(line)
                            if matches:
                                exit_statuses.extend(matches)

                                # Capture timestamp on Exit_status line
                                ts_match = timestamp_re.search(line)
                                if ts_match:
                                    job_endtime = ts_match.group(1)

                        exit_statuses = list(dict.fromkeys(exit_statuses))
                        exit_status_value = ",".join(exit_statuses)

                        # Find last lost communication timestamp
                        last_lost_timestamp = None

                        for line in lines:
                            if "lost communication with" in line.lower():
                                ts_match = timestamp_re.search(line)
                                if ts_match:
                                    last_lost_timestamp = ts_match.group(1)

                        # Compute elapsed minutes
                        elapsed_minutes = ""

                        if last_lost_timestamp and job_endtime:
                            try:
                                t1 = datetime.strptime(last_lost_timestamp, time_format)
                                t2 = datetime.strptime(job_endtime, time_format)
                                elapsed_minutes = round(
                                    (t2 - t1).total_seconds() / 60, 3
                                )
                            except ValueError:
                                elapsed_minutes = ""

                        seen_lost_nodes = set()

                        for line in lost_lines:
                            line = line.strip()

                            lost_match = lost_re.search(line)
                            if not lost_match:
                                continue

                            lost_node = lost_match.group(1)

                            if lost_node in seen_lost_nodes:
                                continue

                            seen_lost_nodes.add(lost_node)

                            timestamp = ""
                            ts_match = timestamp_re.search(line)
                            if ts_match:
                                timestamp = ts_match.group(1)

                            writer.writerow([
                                day_name,
                                job_name,
                                timestamp,
                                last_lost_timestamp,
                                job_endtime,
                                elapsed_minutes,
                                lost_node,
                                kill_status,
                                exit_status_value,
                                line
                            ])

                    except OSError:
                        continue

    except KeyboardInterrupt:
        print("\nScan terminated by user.")

In [6]:
find_lost_communication("/Users/celsloaner/Desktop/Functions/Ping_failures/", "ping_failures_jobtrace.csv")